In [7]:

import matplotlib.pyplot as plt
from src.data.fetcher import fetch_historical
from src.portfolio.simulator import Simulator
from src.agents.agents import MomentumAgent
import yfinance as yf

data = yf.download("AAPL", start="2020-01-01", end="2023-01-01", auto_adjust=True)
#print(data.head())
import pandas as pd

# If you saved to CSV
df = pd.read_csv("data/AAPL.csv", header=[0])  # read first row as header

# If multiindex got saved, collapse it
df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

# Rename columns properly
df.columns = ["Date", "Open", "High", "Low", "Close", "Volume"]

print(df.head())


# Fetch data
df = fetch_historical("AAPL", start="2022-01-01", end="2023-01-01")
# flatten MultiIndex columns
df.columns = [col[0] for col in df.columns]

print(df.head())
print(df.columns)


# Run simulation
agent = MomentumAgent(window=5)
sim = Simulator(df, agent, cash=100000)
results = sim.run()

# Plot portfolio value
results.plot(x="Date", y="PortfolioValue", title="Momentum Backtest", figsize=(10,5))
plt.show()


[*********************100%***********************]  1 of 1 completed

         Date               Open               High                Low  \
0         NaN               AAPL               AAPL               AAPL   
1  2020-01-02  72.53850555419922  72.59888386623527  71.29229630932706   
2  2020-01-03  71.83329772949219  72.59406313642577  71.60869213351766   
3  2020-01-06  72.40567779541016  72.44432080433776   70.7030121336898   
4  2020-01-07  72.06513977050781  72.67133299565539  71.84536183598526   

               Close     Volume  
0               AAPL       AAPL  
1  71.54588227171874  135480400  
2  71.76567430155144  146322800  
3  70.95418800651902  118387200  
4  72.41532954893665  108872000  
           D           A           A           A           A          A
0 2022-01-03  174.345039  179.296076  174.227395  178.443115  104487900
1 2022-01-04  179.050994  179.354917  175.609770  176.178406   99310400
2 2022-01-05  176.090157  176.639180  171.217554  171.492065   94537600
3 2022-01-06  169.315551  171.864605  168.276327  168.629272   

KeyError: 'Close'

In [6]:

import yfinance as yf
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def get_stock(symbol, start="2020-01-01", end="2023-01-01"):
    file_path = DATA_DIR / f"{symbol}.csv"

    if file_path.exists():
        print(f"Loading {symbol} from local file...")
        df = pd.read_csv(file_path, parse_dates=["Date"])
    else:
        print(f"Fetching {symbol} from Yahoo Finance...")
        df = yf.download(symbol, start=start, end=end, auto_adjust=True)
        df = df.reset_index()  # make Date a column
        df.to_csv(file_path, index=False)  # save clean CSV

    return df

if __name__ == "__main__":
    df = get_stock("AAPL")
    print(df.head())


[*********************100%***********************]  1 of 1 completed

Fetching AAPL from Yahoo Finance...
Price        Date      Close       High        Low       Open     Volume
Ticker                  AAPL       AAPL       AAPL       AAPL       AAPL
0      2020-01-02  72.538506  72.598884  71.292296  71.545882  135480400
1      2020-01-03  71.833298  72.594063  71.608692  71.765674  146322800
2      2020-01-06  72.405678  72.444321  70.703012  70.954188  118387200
3      2020-01-07  72.065140  72.671333  71.845362  72.415330  108872000
4      2020-01-08  73.224396  73.526287  71.768071  71.768071  132079200


In [8]:
import pandas as pd
from src.metrics import performance

# fake portfolio values for demo
values = pd.Series([100, 105, 95, 120, 110, 130])

print("Cumulative Returns:", performance.cumulative_returns(values))
print("Annualized Returns:", performance.annualized_returns(values, periods_per_year=252))
print("Sharpe Ratio:", performance.sharpe_ratio(values))
print("Max Drawdown:", performance.max_drawdown(values))


Cumulative Returns: 0.30000000000000004
Annualized Returns: 61039.8815262859
Sharpe Ratio: 6.328021149033161
Max Drawdown: -0.09523809523809523


In [9]:
# Example in Backtester
portfolio = PortfolioManager(initial_cash=100_000)
for i, row in df.iterrows():
    price = row["Close"]
    action = agent.decide_action(row, i, df)
    if action == "BUY":
        portfolio.buy("AAPL", price, 10)
    elif action == "SELL":
        portfolio.sell("AAPL", price, 10)


NameError: name 'PortfolioManager' is not defined